# Adversarial Robustness Experiments on ESM-2
## Paper: Adversarial Robustness and Biosecurity Risks in Protein Language Models

This notebook runs **real experiments** measuring how substitution-based adversarial attacks affect ESM-2's predictions.

### What this notebook does:
1. Loads ESM-2 (esm2_t6_8M_UR50D — small enough for Colab free tier)
2. Establishes **baseline (clean)** perplexity scores
3. Runs substitution attacks at **1%, 5%, 10%** mutation rates
4. Measures perplexity shift after each attack
5. Saves all results to CSV for use in the paper

> **Runtime**: Set to GPU (T4) in Colab for best performance. Runtime > Change runtime type > T4 GPU

## Step 1: Install Dependencies

In [ ]:
# Install required libraries
!pip install fair-esm torch transformers pandas numpy matplotlib seaborn -q
print('Installation complete.')

## Step 2: Import Libraries

In [ ]:
import torch
import esm
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Step 3: Load ESM-2 Model

In [ ]:
# Load ESM-2 smallest variant (8M parameters) — fits on free Colab T4
# For paper: report model as esm2_t6_8M_UR50D
print('Loading ESM-2 model...')
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
model = model.to(device)
model.eval()
batch_converter = alphabet.get_batch_converter()
print('Model loaded successfully.')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 4: Define Protein Sequences
Using well-characterized proteins from UniProt as test sequences.

In [ ]:
# Real protein sequences from UniProt
# These are standard benchmark sequences used in the literature
test_sequences = [
    # Human Ubiquitin (P0CG48) - 76 AA, extremely well characterized
    ("Ubiquitin_P0CG48",
     "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"),

    # Villin headpiece (HP35) - 35 AA, classic folding benchmark
    ("HP35_Villin",
     "LSDEDFKAVFGMTRSAFANLPLWKQQNLKKEKGLF"),

    # Trp-cage miniprotein - 20 AA
    ("Trp_cage",
     "NLYIQWLKDGGPSSGRPPPS"),

    # GB1 protein fragment - 56 AA
    ("GB1_fragment",
     "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE"),

    # Chignolin - 10 AA beta-hairpin
    ("Chignolin",
     "GYDPETGTWG"),

    # WW domain - 34 AA
    ("WW_domain",
     "GSKLPPGWEKRMSRSSGRVYYFNHITNASQWERP"),

    # Human Lysozyme fragment - 50 AA
    ("Lysozyme_fragment",
     "KVFERCELARTLKRLGMDGYRGISLANWMCLAKWESGYNTRATNYNAGDRSTD"),

    # Barnase fragment - 40 AA
    ("Barnase_fragment",
     "AQVINTFDGVADYLQTYHKLPDNYITKSEAQALGWVASK"),
]

print(f'Loaded {len(test_sequences)} protein sequences')
for name, seq in test_sequences:
    print(f'  {name}: {len(seq)} AA')

## Step 5: Define Perplexity Measurement Function
Perplexity measures how 'surprised' the model is by a sequence — higher perplexity after attack = more disruption.

In [ ]:
def compute_perplexity(model, alphabet, batch_converter, sequence, seq_name="protein", device='cpu'):
    """
    Compute pseudo-perplexity of a protein sequence under ESM-2.
    Uses masked marginal scoring: mask each position, measure log-likelihood.
    """
    data = [(seq_name, sequence)]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    batch_tokens = batch_tokens.to(device)

    log_likelihoods = []

    with torch.no_grad():
        for i in range(1, len(sequence) + 1):  # skip <cls> token
            masked_tokens = batch_tokens.clone()
            masked_tokens[0, i] = alphabet.mask_idx

            output = model(masked_tokens, repr_layers=[], return_contacts=False)
            logits = output['logits'][0, i]  # logits at masked position

            # Get log probability of the true amino acid
            true_token = batch_tokens[0, i].item()
            log_prob = torch.log_softmax(logits, dim=-1)[true_token].item()
            log_likelihoods.append(log_prob)

    avg_nll = -np.mean(log_likelihoods)
    perplexity = np.exp(avg_nll)
    return perplexity


# Standard amino acid alphabet
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

def substitution_attack(sequence, mutation_rate, seed=None):
    """
    Random substitution attack: replace amino acids at given mutation rate.
    Each substituted position gets a DIFFERENT amino acid (not the original).
    """
    if seed is not None:
        random.seed(seed)

    seq_list = list(sequence)
    n_mutations = max(1, int(len(sequence) * mutation_rate))
    positions = random.sample(range(len(sequence)), n_mutations)

    for pos in positions:
        original_aa = seq_list[pos]
        # Exclude the original amino acid
        alternatives = [aa for aa in AMINO_ACIDS if aa != original_aa]
        seq_list[pos] = random.choice(alternatives)

    return ''.join(seq_list), positions

print('Functions defined successfully.')

## Step 6: Run Baseline Measurements (Clean Sequences)
**This is the baseline Reviewer 2 specifically asked for.**

In [ ]:
print('Computing baseline perplexity for clean sequences...')
print('(This may take a few minutes)\n')

baseline_results = []

for name, seq in test_sequences:
    ppl = compute_perplexity(model, alphabet, batch_converter, seq, name, device)
    baseline_results.append({
        'sequence_name': name,
        'sequence_length': len(seq),
        'condition': 'clean',
        'mutation_rate': 0.0,
        'perplexity': round(ppl, 4),
        'perplexity_shift': 0.0
    })
    print(f'  {name}: perplexity = {ppl:.4f}')

print('\nBaseline measurements complete.')

## Step 7: Run Adversarial Attacks at 1%, 5%, 10% Mutation Rates

In [ ]:
MUTATION_RATES = [0.01, 0.05, 0.10]
N_TRIALS = 5  # Run each attack 5 times for statistical reliability

attack_results = []
baseline_dict = {r['sequence_name']: r['perplexity'] for r in baseline_results}

print('Running substitution attacks...\n')

for name, seq in test_sequences:
    print(f'Processing: {name}')
    baseline_ppl = baseline_dict[name]

    for rate in MUTATION_RATES:
        trial_perplexities = []

        for trial in range(N_TRIALS):
            attacked_seq, mutated_positions = substitution_attack(seq, rate, seed=SEED + trial)
            ppl = compute_perplexity(model, alphabet, batch_converter, attacked_seq, name, device)
            trial_perplexities.append(ppl)

        mean_ppl = np.mean(trial_perplexities)
        std_ppl = np.std(trial_perplexities)
        ppl_shift = mean_ppl - baseline_ppl
        relative_shift = (ppl_shift / baseline_ppl) * 100

        attack_results.append({
            'sequence_name': name,
            'sequence_length': len(seq),
            'condition': 'attacked',
            'mutation_rate': rate,
            'mutation_rate_pct': f'{int(rate*100)}%',
            'n_mutations': max(1, int(len(seq) * rate)),
            'perplexity_mean': round(mean_ppl, 4),
            'perplexity_std': round(std_ppl, 4),
            'baseline_perplexity': round(baseline_ppl, 4),
            'perplexity_shift': round(ppl_shift, 4),
            'relative_shift_pct': round(relative_shift, 2)
        })

        print(f'  Rate {int(rate*100)}%: mean_ppl={mean_ppl:.4f} (±{std_ppl:.4f}), shift={ppl_shift:+.4f} ({relative_shift:+.1f}%)')

print('\nAll attacks complete.')

## Step 8: Save Results to CSV

In [ ]:
# Save full results
df_attacks = pd.DataFrame(attack_results)
df_attacks.to_csv('esm2_adversarial_results.csv', index=False)

df_baseline = pd.DataFrame(baseline_results)
df_baseline.to_csv('esm2_baseline_results.csv', index=False)

# Summary table for paper
summary = df_attacks.groupby('mutation_rate_pct').agg(
    mean_perplexity=('perplexity_mean', 'mean'),
    std_perplexity=('perplexity_std', 'mean'),
    mean_shift=('perplexity_shift', 'mean'),
    mean_relative_shift=('relative_shift_pct', 'mean')
).reset_index()

print('=== SUMMARY TABLE (for paper) ===')
print(f'Baseline mean perplexity: {df_baseline["perplexity"].mean():.4f}')
print()
print(summary.to_string(index=False))

print('\nFiles saved: esm2_adversarial_results.csv, esm2_baseline_results.csv')

## Step 9: Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('ESM-2 Adversarial Robustness: Substitution Attack Results', fontsize=14, fontweight='bold')

# Plot 1: Perplexity shift by mutation rate
ax1 = axes[0]
rates = ['1%', '5%', '10%']
for name, seq in test_sequences:
    seq_data = df_attacks[df_attacks['sequence_name'] == name]
    shifts = [seq_data[seq_data['mutation_rate_pct'] == r]['perplexity_shift'].values[0] for r in rates]
    ax1.plot(rates, shifts, marker='o', label=name, alpha=0.8)

ax1.set_xlabel('Mutation Rate')
ax1.set_ylabel('Perplexity Shift (attacked - baseline)')
ax1.set_title('Perplexity Shift per Sequence')
ax1.legend(fontsize=7)
ax1.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax1.grid(alpha=0.3)

# Plot 2: Mean relative shift across all sequences
ax2 = axes[1]
mean_shifts = summary['mean_relative_shift'].values
std_shifts = df_attacks.groupby('mutation_rate_pct')['relative_shift_pct'].std().values
bars = ax2.bar(rates, mean_shifts, yerr=std_shifts, capsize=5,
               color=['#4CAF50', '#FF9800', '#F44336'], alpha=0.85)
ax2.set_xlabel('Mutation Rate')
ax2.set_ylabel('Mean Relative Perplexity Shift (%)')
ax2.set_title('Average Impact of Substitution Attacks')
ax2.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, mean_shifts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'+{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('esm2_adversarial_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: esm2_adversarial_results.png')

## Step 10: Generate Paper-Ready Table

In [ ]:
print('=== TABLE FOR PAPER ===')
print('ESM-2 Perplexity Under Substitution Attacks (mean ± std over 5 trials)\n')

baseline_mean = df_baseline['perplexity'].mean()
baseline_std = df_baseline['perplexity'].std()
print(f'Clean (baseline):  {baseline_mean:.2f} ± {baseline_std:.2f}  |  Shift: 0.00 (0.0%)')

for rate in ['1%', '5%', '10%']:
    row = summary[summary['mutation_rate_pct'] == rate].iloc[0]
    print(f'Attack {rate}:         {row["mean_perplexity"]:.2f} ± {row["std_perplexity"]:.2f}  |  '
          f'Shift: +{row["mean_shift"]:.2f} (+{row["mean_relative_shift"]:.1f}%)')

print()
print('Hyperparameters:')
print(f'  Model: esm2_t6_8M_UR50D')
print(f'  Random seed: {SEED}')
print(f'  Trials per condition: {N_TRIALS}')
print(f'  Test sequences: {len(test_sequences)}')
print(f'  Device: {device}')
print(f'  Attack type: Random substitution (non-synonymous)')

## Done!

**Files produced:**
- `esm2_adversarial_results.csv` — full results per sequence per attack rate
- `esm2_baseline_results.csv` — clean baseline perplexity scores
- `esm2_adversarial_results.png` — figure for the paper

**What to report in the paper:**
1. Use the baseline perplexity row — this directly addresses Reviewer 2's request
2. Report mean ± std across trials — shows statistical rigor
3. Include this notebook URL as your public code repository — addresses Reviewer 1's request
4. State: model = esm2_t6_8M_UR50D, seed = 42, 5 trials per condition

**Next step:** Share this notebook on Google Colab and copy the shareable link — that becomes your public code repository.